# Question 1: Shortest Common Superstring (SCS)

In a practical, we saw the `scs` function (copied below along with `overlap`) for finding the shortest common superstring of a set of strings.

It is possible for there to be multiple different shortest common superstrings for the same set of input strings. For example, consider the input strings:

- `ABC`
- `BCA`
- `CAB`

One shortest common superstring is `ABCAB`, another is `BCABC`, and another is `CABCA`.

**Question:** What is the length of the shortest common superstring of the following strings?

- `CCT`
- `CTT`
- `TGC`
- `TGG`
- `GAT`
- `ATT`


In [2]:
import itertools

def overlap(a, b, min_length=1):
    """Return length of longest suffix of a matching a prefix of b."""
    start = 0
    while True:
        start = a.find(b[:min_length], start)
        if start == -1:
            return 0
        if b.startswith(a[start:]):
            return len(a) - start
        start += 1

def scs(ss):
    """Return one shortest common superstring."""
    shortest_sup = None

    for ssperm in itertools.permutations(ss):
        sup = ssperm[0]

        for i in range(len(ssperm) - 1):
            olen = overlap(ssperm[i], ssperm[i + 1], min_length=1)
            sup += ssperm[i + 1][olen:]

        if shortest_sup is None or len(sup) < len(shortest_sup):
            shortest_sup = sup

    return shortest_sup

strings = ["CCT", "CTT", "TGC", "TGG", "GAT", "ATT"]

answer = scs(strings)

print(answer)
print(len(answer))

CCTTGGATTGC
11


# Question 2: Count Different Shortest Common Superstrings

How many different shortest common superstrings are there for the input strings given in the previous question?

Input strings:

```python
["CCT", "CTT", "TGC", "TGG", "GAT", "ATT"]
```

In [3]:
import itertools

def overlap(a, b, min_length=1):
    """Return length of longest suffix of a matching a prefix of b."""
    start = 0
    while True:
        start = a.find(b[:min_length], start)
        if start == -1:
            return 0
        if b.startswith(a[start:]):
            return len(a) - start
        start += 1

def scs_all(ss):
    """Return all different shortest common superstrings."""
    shortest_len = None
    shortest_sups = set()

    for ssperm in itertools.permutations(ss):
        sup = ssperm[0]

        for i in range(len(ssperm) - 1):
            olen = overlap(ssperm[i], ssperm[i + 1], min_length=1)
            sup += ssperm[i + 1][olen:]

        if shortest_len is None or len(sup) < shortest_len:
            shortest_len = len(sup)
            shortest_sups = {sup}
        elif len(sup) == shortest_len:
            shortest_sups.add(sup)

    return shortest_sups

strings = ["CCT", "CTT", "TGC", "TGG", "GAT", "ATT"]

shortest_sups = scs_all(strings)

print("Number of different shortest common superstrings:", len(shortest_sups))
print("Length:", len(next(iter(shortest_sups))))

for s in sorted(shortest_sups):
    print(s)

Number of different shortest common superstrings: 4
Length: 11
CCTTGGATTGC
GATTGCCTTGG
TGCCTTGGATT
TGGATTGCCTT


# Question 3: Assemble FASTQ Reads from a Mystery Virus

Download the FASTQ file containing synthetic sequencing reads from a mystery virus:

```text
https://d28rh4a8wq0iu5.cloudfront.net/ads1/data/ads1_week4_reads.fq
```

All reads are the same length, 100 bases, and are exact copies of substrings from the forward strand of the virus genome.

There are no sequencing errors, no ploidy issues, and no reverse-strand reads.

Assemble these reads using one of the approaches discussed, such as greedy shortest common superstring.

**Question:** How many `A`s are there in the full assembled genome?

**Hint:** The virus genome is exactly **15,894 bases** long.

# Question 4: Count T Bases in the Assembled Genome

How many `T`s are there in the full, assembled genome from the previous question?

In [18]:
from pathlib import Path
from collections import defaultdict

def readFastq(filename):
    reads = []
    with open(filename) as f:
        while True:
            name = f.readline()
            if not name:
                break
            seq = f.readline().strip()
            f.readline()
            f.readline()
            reads.append(seq)
    return reads


def overlap(a, b, min_length=3):
    start = 0
    while True:
        start = a.find(b[:min_length], start)
        if start == -1:
            return 0
        if b.startswith(a[start:]):
            return len(a) - start
        start += 1


def build_kmer_index(reads, k):
    index = defaultdict(set)

    for read in reads:
        for i in range(len(read) - k + 1):
            kmer = read[i:i+k]
            index[kmer].add(read)

    return index


def pick_maximal_overlap_fast(reads, k):
    index = build_kmer_index(reads, k)

    best_a = None
    best_b = None
    best_olen = 0

    for a in reads:
        suffix = a[-k:]

        for b in index[suffix]:
            if a != b:
                olen = overlap(a, b, min_length=k)

                if olen > best_olen:
                    best_a = a
                    best_b = b
                    best_olen = olen

    return best_a, best_b, best_olen


def greedy_scs_fast(reads, k=30):
    reads = list(reads)

    while True:
        a, b, olen = pick_maximal_overlap_fast(reads, k)

        if olen == 0:
            break

        reads.remove(a)
        reads.remove(b)
        reads.append(a + b[olen:])

    return "".join(reads)


# File path on Mac Downloads folder
fastq_file = Path.home() / "Downloads" / "ads1_week4_reads.fq"

reads = readFastq(fastq_file)

genome = greedy_scs_fast(reads, k=30)

print("Genome length:", len(genome))
print("A count:", genome.count("A"))
print("T count:", genome.count("T"))

Genome length: 15894
A count: 4633
T count: 3723
